In [10]:
trainer.evaluate()


{'eval_loss': 0.24552009999752045,
 'eval_runtime': 262.7372,
 'eval_samples_per_second': 7.612,
 'eval_steps_per_second': 0.476,
 'epoch': 1.0}

In [9]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./smoke_test",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    logging_steps=50,
    save_strategy="no",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    tokenizer=tokenizer
)

trainer.train()


/tmp/ipython-input-3092279114.py:13: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
50,0.692900
100,0.669300
150,0.626900
200,0.482800
250,0.415900
300,0.364900
350,0.354200
400,0.329600
450,0.307300
500,0.346900


TrainOutput(global_step=625, training_loss=0.42663106079101565, metrics={'train_runtime': 5739.2056, 'train_samples_per_second': 1.742, 'train_steps_per_second': 0.109, 'total_flos': 165829048320000.0, 'train_loss': 0.42663106079101565, 'epoch': 1.0})

In [7]:
def tokenize(batch):
    return tokenizer(
        batch["text_A"],
        batch["text_B"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

train_ds = train_ds.map(tokenize, batched=True)
valid_ds = valid_ds.map(tokenize, batched=True)

train_ds = train_ds.remove_columns(["text_A", "text_B"])
valid_ds = valid_ds.remove_columns(["text_A", "text_B"])

train_ds.set_format("torch")
valid_ds.set_format("torch")


Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [6]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at sentence-transformers/all-MiniLM-L6-v2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:
from datasets import Dataset

train_ds = Dataset.from_pandas(train_df)
valid_ds = Dataset.from_pandas(valid_df)


In [4]:
TRAIN_PATH = "/content/drive/MyDrive/medical/capstone_work/train.csv"
VALID_PATH = "/content/drive/MyDrive/medical/capstone_work/valid.csv"

train_df = pd.read_csv(TRAIN_PATH, nrows=10_000)
valid_df = pd.read_csv(VALID_PATH, nrows=2_000)

print(train_df.shape, valid_df.shape)
print(train_df["label"].value_counts())



(10000, 3) (2000, 3)
label
1    5039
0    4961
Name: count, dtype: int64


0. 환경준비

In [3]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
import torch.nn as nn


In [2]:
!pip install -q transformers datasets accelerate sentence-transformers


드라이브 마운트

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
